# 03. Tuning bobot: dicari dari data, bukan dikarang

**Pertanyaan:** bobot lima sinyal teks di `ai_score.py` semula ditulis dari penalaran sebelum ada pengukuran.
Bisakah bobot dicari dari data tanpa membocorkan test?

Protokol yang membuat angkanya layak dipercaya:

1. **Pencarian hanya melihat split train.** Split test tidak disentuh selama pencarian; angka test yang boleh dikutip.
2. **Lantai bobot 0,05** untuk tiga sinyal yang diam di register abstrak (`formulaic_phrasing`,
   `impersonality`, `flat_certainty`). Diam di register yang salah bukan bukti buruk di register esai
   mahasiswa yang sebenarnya dinilai; menolkannya berarti membuang sinyal yang belum pernah diuji.
3. **Ambang band diturunkan ulang** untuk bobot terpilih. Ambang diukur untuk sebaran skor satu set bobot
   tertentu; mengganti bobot tanpa mengulang pencarian ambang meninggalkan angka tanpa dasar ukur.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src import config

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 140)
config.ensure_dirs()
GOLD = config.GOLD_DIR / "gold_set.csv"

In [2]:
from src import evaluate_baseline as eb
from src import tune_weights as tw

eb.bootstrap_backend()

grid_bebas = tw.weight_grid({})
grid_lantai = tw.weight_grid({key: 0.05 for key in tw.UNTESTED_KEYS})
pd.Series(
    {"kombinasi bobot (langkah 0,05, jumlah 1,0), pencarian bebas": len(grid_bebas),
     "kombinasi dengan lantai 0,05 untuk 3 sinyal": len(grid_lantai)},
    name="ukuran ruang pencarian",
).to_frame()

,ukuran ruang pencarian
"kombinasi bobot (langkah 0,05, jumlah 1,0), pencarian bebas",10626
"kombinasi dengan lantai 0,05 untuk 3 sinyal",5985


## Tahap 1. Pisahkan train dan test

In [3]:
HAS_GOLD = GOLD.exists()
if not HAS_GOLD:
    print("gold_set.csv belum ada (notebook 01). Sel-sel di bawah dilewati.")
else:
    rows = eb.load_rows(GOLD, None, "exclude")
    train, test = tw.split_rows(rows)
    print(f"{len(rows)} sampel: {len(train)} train (dipakai mencari), {len(test)} test (hanya untuk menguji)")

gold_set.csv belum ada (notebook 01). Sel-sel di bawah dilewati.


## Tahap 2. Jalankan protokol

`run_tuning` menghitung kelima sinyal produksi untuk semua sampel, mencari bobot terbaik di train
(bebas, lalu dengan lantai), dan mengukur ketiga kandidat (bobot sekarang, bebas, lantai) di test.

In [4]:
if HAS_GOLD:
    hasil = tw.run_tuning(rows)
    kandidat = pd.DataFrame(hasil["candidates"])
    bobot = pd.DataFrame({k["kandidat"]: k["bobot"] for k in hasil["candidates"]})
    display(kandidat.drop(columns="bobot").set_index("kandidat").round(3))
    display(bobot.round(2))

In [5]:
if HAS_GOLD:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
    pd.Series(hasil["solo_auc"]).sort_values().plot.barh(ax=axes[0], color="#0E7C86")
    axes[0].axvline(0.5, color="#444", linestyle=":")
    axes[0].set(title="AUC tiap sinyal sendirian (test)", xlim=(0.3, 1.0))
    kandidat.set_index("kandidat")[["auc_train", "auc_test"]].plot.bar(ax=axes[1], rot=0, color=["#9AB8BB", "#0E7C86"])
    axes[1].set(title="AUC: train vs test", ylim=(0.5, 1.0), xlabel="")
    bobot.T.plot.bar(stacked=True, ax=axes[2], rot=0, legend=False)
    axes[2].set(title="Bobot per kandidat", xlabel="")
    axes[2].legend(fontsize=7, bbox_to_anchor=(1.0, 1.0))
    plt.tight_layout()
    plt.show()

## Tahap 3. Keputusan

Yang dibandingkan adalah **AUC test** (bukan train: train sengaja dihafal pencarian) dan ambang band yang
menjaga FPR di bawah 5%.

* Varian **lantai 0,05** yang layak masuk produksi, bukan varian bebas yang menolkan sinyal tak teruji.
* Selisih AUC test (lantai vs sekarang) = `kandidat.auc_test[lantai] - kandidat.auc_test[sekarang]`.
* Bobot terpilih sudah dipakai di `backend/apps/analysis/engines/ai_score.py`
  (`TEXT_WEIGHTS`, `MID_THRESHOLD = 42`). **Mengubah bobot wajib menurunkan ulang ambang di notebook ini.**

In [6]:
if HAS_GOLD:
    sekarang, lantai = kandidat.set_index("kandidat").loc["sekarang"], kandidat.set_index("kandidat").loc["lantai 0,05"]
    print(f"Selisih AUC test (lantai vs sekarang): {lantai.auc_test - sekarang.auc_test:+.3f}")
    print(f"Ambang aman: {int(sekarang.ambang_fpr5)} → {int(lantai.ambang_fpr5)}; recall {sekarang.recall:.3f} → {lantai.recall:.3f}")

## Kesimpulan

**Tercatat 17 Agustus 2026** (gold set 999 sampel, 774 train / 225 test, abstrak akademik):

| | sebelum | sesudah |
|---|---|---|
| ROC-AUC test | 0,884 | **0,912** |
| ROC-AUC set penuh | 0,869 | **0,901** |
| ambang `mid` (FPR < 5%) | 56 | **42** |
| recall pada ambang itu | 0,489 | 0,607 |

Catatan register: seluruh angka diukur pada abstrak. Sinyal yang diam di register ini belum tentu diam pada
esai mahasiswa.

Lanjut ke `04_rantai_llm_dan_detektor`.